# Study 961 — Which Gold — the teardown

Realised tracking differences on a five-wrapper, single-asset cross-section: the three estimators and why they disagree, the measurement floor, the fee-rank test against an exactly enumerated permutation null, the pass-through regression, the cheapest-minus-priciest spread with **naive and HAC *t*s and two bootstraps**, an era cut, a fee-stable sub-window, the excess-of-cash ownership race with one execution lag, four swept counterweights, and the live synthetic control.

**Read the naive *t*, not the HAC one.** The wrapper-versus-wrapper residual mean-reverts at both frequencies, so Newey-West *shrinks* the long-run variance here and the HAC *t* runs about twice the iid one — and keeps growing with the bandwidth. Same logic on the interval: the block bootstrap inherits the mean reversion, so the **iid** bootstrap CI is the one quoted as the honest one.

Every real number is frozen from `docs/results.md` (Fingerprint `9af52c692ea9`); the synthetic cells are labelled and never sit under a real-tape banner.

In [1]:
R = {'start': '2018-06-26', 'end': '2026-06-30', 'n_days': 2013, 'n_months': 96, 'fp': '9af52c692ea9', 'fees': {'GLD': 40.0, 'IAU': 25.0, 'GLDM': 10.0, 'SGOL': 17.0, 'BAR': 17.49}, 'td': {'GLD': -16.75, 'IAU': -2.24, 'GLDM': 9.43, 'SGOL': 5.41, 'BAR': 4.15}, 'td_t': {'GLD': -3.02, 'IAU': -0.31, 'GLDM': 1.8, 'SGOL': 0.56, 'BAR': 0.54}, 'td_t_hac': {'GLD': -6.95, 'IAU': -0.62, 'GLDM': 3.45, 'SGOL': 1.04, 'BAR': 1.02}, 'td_up': {'GLD': 32, 'IAU': 47, 'GLDM': 59, 'SGOL': 49, 'BAR': 51}, 'spearman': -1.0, 'p_perm': 0.0167, 'p_floor': 0.0167, 'n_perm': 120, 'pt_slope': -0.894, 'pt_r2': 0.992, 'pt_slope_blend': -0.943, 'pt_r2_blend': 0.999, 'pair': 'GLDM - GLD', 'spread': 26.17, 'spread_t': 3.18, 'spread_t_hac': 6.32, 'spread_up': 65, 'spread_sd_m': 6.7, 'spread_acf1': -0.424, 'pos_years': 9, 'n_years': 9, 'ci_lo': 9.54, 'ci_hi': 41.88, 'ci_neg': 0.06, 'ci_blk_lo': 17.09, 'ci_blk_hi': 35.28, 'ci_blk_neg': 0.0, 'hac_sweep': [('naive', 3.18), ('lag 1', 4.22), ('lag 3', 6.32), ('lag 12', 9.43)], 'daily': 26.36, 'daily_t': 2.19, 'daily_t_naive': 0.78, 'daily_sd': 5.98, 'daily_acf1': -0.462, 'endpoint': 26.28, 'fee_gap': 30.0, 'fee_gap_blend': 27.74, 'floor_bpy': 22.5, 'floor_sd_d': 7.43, 'floor_sd_m': 9.2, 'era_e_n': 42, 'era_e': 20.98, 'era_e_t': 1.43, 'era_e_t_hac': 2.87, 'era_e_up': 23, 'era_l_n': 54, 'era_l': 30.21, 'era_l_t': 3.28, 'era_l_t_hac': 6.62, 'era_l_up': 42, 'stable_n': 65, 'stable': 26.66, 'stable_t': 2.69, 'stable_t_hac': 5.24, 'stable_up': 47, 'stable_rho': -1.0, 'cost1': 0.262, 'cost5': 1.302, 'cost10': 2.587, 'cost20': 5.107, 'per100k10': 2587, 'sh_cheap': 0.7799, 'sh_dear': 0.7613, 'sh_chase': 0.7762, 'cagr_cheap': 15.476, 'cagr_dear': 15.172, 'cagr_chase': 15.427, 'sh_gap': 0.0187, 'cd_t': 0.74, 'cd_t_hac': 2.06, 'chase_t': -0.14, 'chase_t_hac': -0.41, 'n_switches': 4, 'adv': {'GLD': 4085, 'IAU': 654, 'GLDM': 433, 'SGOL': 175, 'BAR': 21}, 'adv_days': {'GLD': 0.02, 'IAU': 0.15, 'GLDM': 0.23, 'SGOL': 0.57, 'BAR': 4.8}, 'be5': 69.7, 'be10': 139.5, 'be30': 418.4, 'tax_20_50': 26.4, 'tax_238_300': 75.1, 'ls_gross': 18.17, 'ls_dead_above': 18.0, 'syn_gap': 30.0, 'syn_rec': 30.98, 'syn_t': 2.55, 'syn_rho': -1.0, 'syn_slope': -1.041, 'syn_null_mean': -0.47, 'syn_null_sd': 1.07, 'syn_null_fire': 0, 'syn_null_p': 0}

## 1. The panel and its non-tape inputs

> 💡 **In plain words:** the fee sheet is the only thing in this study that is not measured. Everything with a *t* next to it is pure tape — but the fee sheet still chose which pair to point the tape at.

Daily adjusted closes, `auto_adjust=True`. None of the five grantor trusts distributes — they sell bullion to pay the sponsor — so **adjusted close = price close** and the total-return / price-only labels coincide. BIL is genuine total return and appears only as the cash leg. **Survivorship:** these are five wrappers that still exist; closed bullion vehicles are absent by construction, which flatters the cohort *level* and cannot manufacture the cross-sectional *ordering*.

ASSUMPTIONS, all labelled and swept where they matter: *today's* published fee sheet (hindsight-carrying — on day one of the window GLDM charged 18 bp and was not the cheapest of the five), GLDM's 18 → 10 bp cut on 2020-10-01 (an announcement date, invisible on a price tape, and the only change this study records), the round-trip execution differential, the borrow rate, and the capital-gains rate. One **selection** on top of those: the headline pair is chosen off that sheet — the number is pure tape, the choice of legs is not.

## 2. The measurement floor — what this tape can and cannot resolve

In [2]:
print(f"pairwise noise: {R['floor_sd_d']:.2f} bp/day, {R['floor_sd_m']:.1f} bp/month "
      f"over {R['n_months']} months")
print(f"smallest gap separable from zero at |t|=2: {R['floor_bpy']:.1f} bp/yr")
print(f"published fee spread across the cohort:    {R['fee_gap']:.1f} bp/yr")
print('-> the coarse ranking is resolvable; the single-digit rungs are not')

pairwise noise: 7.43 bp/day, 9.2 bp/month over 96 months
smallest gap separable from zero at |t|=2: 22.5 bp/yr
published fee spread across the cohort:    30.0 bp/yr
-> the coarse ranking is resolvable; the single-digit rungs are not


## 3. Cohort-relative tracking difference, three estimators

Each wrapper minus the equal-weight cohort mean: the bullion price and any common strike effect cancel exactly, leaving the fee plus each wrapper's own premium/discount.

In [3]:
print(f"{'fund':6s} {'fee':>6s} {'monthly':>9s} {'t':>7s} {'[HAC]':>8s} {'up':>8s}")
for t in ['GLDM', 'SGOL', 'BAR', 'IAU', 'GLD']:
    print(f"{t:6s} {R['fees'][t]:6.2f} {R['td'][t]:+9.2f} {R['td_t'][t]:+7.2f} "
          f"{R['td_t_hac'][t]:+8.2f} {R['td_up'][t]:5d}/{R['n_months']}")
print(f"\ncross-section sums to ~0 by construction: {sum(R['td'].values()):+.2f} bp/yr")
print('on the honest t only GLD, the priciest, separates from the cohort mean alone;')
print('GLDM is the right sign and size at t=+1.80 -- a 9 bp effect against a 22.5 bp')
print('floor. The evidence is the widest PAIR and the JOINT ranking, not one wrapper.')

fund      fee   monthly       t    [HAC]       up
GLDM    10.00     +9.43   +1.80    +3.45    59/96
SGOL    17.00     +5.41   +0.56    +1.04    49/96
BAR     17.49     +4.15   +0.54    +1.02    51/96
IAU     25.00     -2.24   -0.31    -0.62    47/96
GLD     40.00    -16.75   -3.02    -6.95    32/96

cross-section sums to ~0 by construction: +0.00 bp/yr
on the honest t only GLD, the priciest, separates from the cohort mean alone;
GLDM is the right sign and size at t=+1.80 -- a 9 bp effect against a 22.5 bp
floor. The evidence is the widest PAIR and the JOINT ranking, not one wrapper.


## 4. The fee-rank test and the pass-through regression

> 💡 **In plain words:** the first asks whether the fee *order* predicts the outcome *order*; the second asks whether a basis point of fee costs a basis point of return.

In [4]:
print(f"Spearman(fee, tracking difference) = {R['spearman']:+.4f}")
print(f"exact permutation p = {R['p_perm']:.4f} over all {R['n_perm']} permutations")
print(f"  ...which is the HARD FLOOR: with 5 funds no match, however perfect,")
print(f"     can print below p = {R['p_floor']:.4f}. The rank test is at its ceiling.")
print(f"\npass-through slope (today's sheet): {R['pt_slope']:+.3f}  R2 {R['pt_r2']:.3f}")
print(f"pass-through slope (time-blended) : {R['pt_slope_blend']:+.3f}  R2 {R['pt_r2_blend']:.3f}")
print('-> a basis point of fee costs ~0.9 of a basis point of return, and the fee')
print('   sheet explains 99% of a five-point cross-section')

Spearman(fee, tracking difference) = -1.0000
exact permutation p = 0.0167 over all 120 permutations
  ...which is the HARD FLOOR: with 5 funds no match, however perfect,
     can print below p = 0.0167. The rank test is at its ceiling.

pass-through slope (today's sheet): -0.894  R2 0.992
pass-through slope (time-blended) : -0.943  R2 0.999
-> a basis point of fee costs ~0.9 of a basis point of return, and the fee
   sheet explains 99% of a five-point cross-section


## 5. The headline pair — GLDM (10 bp) minus GLD (40 bp)

No fee number enters this measurement. **Which** two wrappers get differenced is read off the fee sheet, though — the pair is a selection, and it is the widest gap on the sheet, i.e. the pair with the most to find.

In [5]:
print(f"monthly (non-overlapping): {R['spread']:+.2f} bp/yr  naive t {R['spread_t']:+.2f}  "
      f"[HAC {R['spread_t_hac']:+.2f}]  {R['spread_up']}/{R['n_months']} months up  "
      f"sd {R['spread_sd_m']:.1f} bp/mo  acf1 {R['spread_acf1']:+.3f}")
print(f"positive in {R['pos_years']}/{R['n_years']} calendar years")
print(f"iid bootstrap   95% CI   : [{R['ci_lo']:+.2f}, {R['ci_hi']:+.2f}] bp/yr  "
      f"share<0 {R['ci_neg']:.2f}%   <- the conservative interval")
print(f"block bootstrap 95% CI   : [{R['ci_blk_lo']:+.2f}, {R['ci_blk_hi']:+.2f}] bp/yr  "
      f"share<0 {R['ci_blk_neg']:.2f}%   (inherits the mean reversion)")
print(f"daily                    : {R['daily']:+.2f} bp/yr  naive t {R['daily_t_naive']:+.2f}  "
      f"[HAC {R['daily_t']:+.2f}]")
print(f"endpoint                 : {R['endpoint']:+.2f} bp/yr")
print(f"\ndaily residual: sd {R['daily_sd']:.2f} bp, acf1 {R['daily_acf1']:+.3f}")
print('HAC bandwidth sweep: ' + '  '.join('%s=%+.2f' % (k, v) for k, v in R['hac_sweep']))
print('-> the residual MEAN-REVERTS at BOTH frequencies, so the HAC long-run variance')
print('   sits BELOW the iid one: the HAC t is the ANTI-conservative choice and it')
print('   keeps climbing with the bandwidth. Monthly aggregation is the right')
print('   construction (non-overlapping observations) but it is NOT a conservative')
print('   one, and this study does not pretend otherwise: it quotes the naive t.')
print(f"\nrealised {R['spread']:.2f} vs sheet gap {R['fee_gap']:.2f}; time-blending GLDM's")
print(f"18 bp launch fee expects {R['fee_gap_blend']:.2f} -> about half the shortfall is fee")
print('history and the rest sits well inside the iid bootstrap half-width (+/-16 bp).')

monthly (non-overlapping): +26.17 bp/yr  naive t +3.18  [HAC +6.32]  65/96 months up  sd 6.7 bp/mo  acf1 -0.424
positive in 9/9 calendar years
iid bootstrap   95% CI   : [+9.54, +41.88] bp/yr  share<0 0.06%   <- the conservative interval
block bootstrap 95% CI   : [+17.09, +35.28] bp/yr  share<0 0.00%   (inherits the mean reversion)
daily                    : +26.36 bp/yr  naive t +0.78  [HAC +2.19]
endpoint                 : +26.28 bp/yr

daily residual: sd 5.98 bp, acf1 -0.462
HAC bandwidth sweep: naive=+3.18  lag 1=+4.22  lag 3=+6.32  lag 12=+9.43
-> the residual MEAN-REVERTS at BOTH frequencies, so the HAC long-run variance
   sits BELOW the iid one: the HAC t is the ANTI-conservative choice and it
   keeps climbing with the bandwidth. Monthly aggregation is the right
   construction (non-overlapping observations) but it is NOT a conservative
   one, and this study does not pretend otherwise: it quotes the naive t.

realised 26.17 vs sheet gap 30.00; time-blending GLDM's
18 bp laun

## 6. Era cut and the fee-stable sub-window

Two ways of asking whether this is one lucky stretch.

In [6]:
print(f"2018-07 -> 2021-12 (n={R['era_e_n']}): {R['era_e']:+.2f} bp/yr  "
      f"t={R['era_e_t']:+.2f} [HAC {R['era_e_t_hac']:+.2f}]  {R['era_e_up']}/{R['era_e_n']} up")
print(f"2022-01 -> 2026-06 (n={R['era_l_n']}): {R['era_l']:+.2f} bp/yr  "
      f"t={R['era_l_t']:+.2f} [HAC {R['era_l_t_hac']:+.2f}]  {R['era_l_up']}/{R['era_l_n']} up")
print('-> positive in BOTH halves, significant on the honest t in the LATE one only.')
print('   42 months of a ~21 bp effect against a 22.5 bp floor CANNOT be significant,')
print('   so the era cut buys sign-stability, not two independent confirmations.')
print('   The early era is smaller by about what GLDM 18 bp (until 2020-10-01)')
print(f"   implies, and the late era prints {R['era_l']:.2f} against a {R['fee_gap']:.2f} bp fee gap.")
print(f"\nfee-stable sub-window (2021-01 ->, n={R['stable_n']} months): "
      f"{R['stable']:+.2f} bp/yr  t={R['stable_t']:+.2f} [HAC {R['stable_t_hac']:+.2f}]  "
      f"{R['stable_up']}/{R['stable_n']} up  Spearman {R['stable_rho']:+.4f}")
print('-> the cut that matters most: no fee assumption does any work in it, and it')
print('   clears |t|=2 on its own.')

2018-07 -> 2021-12 (n=42): +20.98 bp/yr  t=+1.43 [HAC +2.87]  23/42 up
2022-01 -> 2026-06 (n=54): +30.21 bp/yr  t=+3.28 [HAC +6.62]  42/54 up
-> positive in BOTH halves, significant on the honest t in the LATE one only.
   42 months of a ~21 bp effect against a 22.5 bp floor CANNOT be significant,
   so the era cut buys sign-stability, not two independent confirmations.
   The early era is smaller by about what GLDM 18 bp (until 2020-10-01)
   implies, and the late era prints 30.21 against a 30.00 bp fee gap.

fee-stable sub-window (2021-01 ->, n=65 months): +26.66 bp/yr  t=+2.69 [HAC +5.24]  47/65 up  Spearman -1.0000
-> the cut that matters most: no fee assumption does any work in it, and it
   clears |t|=2 on its own.


## 7. The ownership race, excess-of-cash — one execution lag, costs × NAV

> 💡 **In plain words:** 26 bp a year on a 17%-vol asset is worth about two hundredths of a Sharpe point. Real, and invisible in a performance table.

In [7]:
print(f"own cheapest (GLDM): exSharpe {R['sh_cheap']:.4f}  CAGR {R['cagr_cheap']:+.3f}%")
print(f"own priciest (GLD) : exSharpe {R['sh_dear']:.4f}  CAGR {R['cagr_dear']:+.3f}%")
print(f"chase last winner  : exSharpe {R['sh_chase']:.4f}  CAGR {R['cagr_chase']:+.3f}%  "
      f"({R['n_switches']} switches, 2 bp one-way x NAV)")
print(f"\nSharpe gap cheap-dear {R['sh_gap']:+.4f}  -- and NO t is quoted on that gap.")
print(f"daily return diff cheap-dear: naive t {R['cd_t']:+.2f} [HAC {R['cd_t_hac']:+.2f}]")
print('   the daily view is powerless in both directions; the monthly estimator above')
print('   is the evidence, and this race only shows what 26 bp/yr LOOKS like in a')
print('   performance table: two hundredths of a Sharpe point. Real, and invisible.')
print(f"chase vs own-cheapest: t {R['chase_t']:+.2f} [HAC {R['chase_t_hac']:+.2f}]  "
      f"-> the persistence rule LOSES")
print('\nlag: ranked on data through the last session of the calendar year, FILLED at')
print('the next session close, so the new wrapper first earns on the session after')
print('that -- never a same-close fill. That is the only decision in the study, and')
print('the only place a lag or a cost can apply. No short leg here, so no borrow.')

own cheapest (GLDM): exSharpe 0.7799  CAGR +15.476%
own priciest (GLD) : exSharpe 0.7613  CAGR +15.172%
chase last winner  : exSharpe 0.7762  CAGR +15.427%  (4 switches, 2 bp one-way x NAV)

Sharpe gap cheap-dear +0.0187  -- and NO t is quoted on that gap.
daily return diff cheap-dear: naive t +0.74 [HAC +2.06]
   the daily view is powerless in both directions; the monthly estimator above
   is the evidence, and this race only shows what 26 bp/yr LOOKS like in a
   performance table: two hundredths of a Sharpe point. Real, and invisible.
chase vs own-cheapest: t -0.14 [HAC -0.41]  -> the persistence rule LOSES

lag: ranked on data through the last session of the calendar year, FILLED at
the next session close, so the new wrapper first earns on the session after
that -- never a same-close fill. That is the only decision in the study, and
the only place a lag or a cost can apply. No short leg here, so no borrow.


## 8. The four counterweights, all swept

In [8]:
print('LIQUIDITY (real tape, close x volume, last 12 months):')
for t in ['GLDM', 'SGOL', 'BAR', 'IAU', 'GLD']:
    print(f"  {t:6s} ${R['adv'][t]:6,}m/day   a $100m ticket = {R['adv_days'][t]:5.2f} days of ADV")
print(f"\nEXECUTION differential (ASSUMPTION, swept): 5 bp round trip repaid in "
      f"{R['be5']:.0f} days, 10 bp in {R['be10']:.0f}, a punitive 30 bp in {R['be30']:.0f}")
print(f"TAX on switching (ASSUMPTION, swept): 20% on a doubled position takes "
      f"{R['tax_20_50']:.1f} years;")
print(f"   23.8% collectibles rate on a 4x position takes {R['tax_238_300']:.1f} years")
print(f"LONG/SHORT (ASSUMPTION, swept): gross {R['spread']:.2f} bp/yr becomes "
      f"{R['ls_gross']:.2f} after costs")
print(f"   and dies above ~{R['ls_dead_above']:.0f} bp/yr of borrow -> not a trade")

LIQUIDITY (real tape, close x volume, last 12 months):
  GLDM   $   433m/day   a $100m ticket =  0.23 days of ADV
  SGOL   $   175m/day   a $100m ticket =  0.57 days of ADV
  BAR    $    21m/day   a $100m ticket =  4.80 days of ADV
  IAU    $   654m/day   a $100m ticket =  0.15 days of ADV
  GLD    $ 4,085m/day   a $100m ticket =  0.02 days of ADV

EXECUTION differential (ASSUMPTION, swept): 5 bp round trip repaid in 70 days, 10 bp in 140, a punitive 30 bp in 418
TAX on switching (ASSUMPTION, swept): 20% on a doubled position takes 26.4 years;
   23.8% collectibles rate on a 4x position takes 75.1 years
LONG/SHORT (ASSUMPTION, swept): gross 26.17 bp/yr becomes 18.17 after costs
   and dies above ~18 bp/yr of borrow -> not a trade


## 9. Live synthetic control — the machinery is unbiased

**Synthetic, not the real tape.** Planted ladder: the stack MUST recover it. Null (every wrapper charging the cohort-average fee while the published sheet still shows dispersion): it MUST stay quiet.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from which_gold import data, strategy as st
pl_p, pl_t = data.synthetic_panel(signal_strength=1.0, seed=961)
pl = st.synthetic_detect(pl_p, pl_t)
print('SYNTHETIC planted: gap %.1f -> recovered %+.2f bp/yr (t=%+.2f), Spearman %+.3f '
      '(p=%.4f), pass-through %+.3f (R2 %.3f)'
      % (pl['planted_gap_bpy'], pl['pair_spread_bpy'], pl['pair_t'], pl['spearman'],
         pl['p_perm'], pl['pass_through_slope'], pl['pass_through_r2']))
nl = [st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0, seed=961+s)) for s in range(8)]
sp = np.array([n['pair_spread_bpy'] for n in nl])
tt = np.array([n['pair_t'] for n in nl])
pp = np.array([n['p_perm'] for n in nl])
print('SYNTHETIC null x8: spread mean %+.2f bp/yr (sd %.2f), |t|>=2 in %d/8, p<0.05 in %d/8'
      % (sp.mean(), sp.std(ddof=1), int((abs(tt) >= 2).sum()), int((pp < 0.05).sum())))

SYNTHETIC planted: gap 30.0 -> recovered +30.98 bp/yr (t=+2.55), Spearman -1.000 (p=0.0167), pass-through -1.041 (R2 0.999)


SYNTHETIC null x8: spread mean -0.47 bp/yr (sd 1.07), |t|>=2 in 0/8, p<0.05 in 0/8


## Verdict

- **Signal — Real.** Cheapest minus priciest is **+26.17 bp/yr, naive *t* = +3.18** (HAC +6.32, and inflated — see the bandwidth sweep), **iid** bootstrap CI **[+9.5, +41.9]** clear of zero, 65/96 months and 9/9 calendar years positive, positive in both eras (+21.0 / +30.2, significant in the late one) and **+26.7 at *t* = +2.69 on the fee-stable sub-window**, the one cut in which no fee assumption does any work. The fee ranking inverts the outcome ranking **perfectly** (Spearman -1.0000, at its 0.0167 permutation floor) with pass-through **-0.89 to -0.94**, R² **0.99** and ten of ten pairwise signs correct; the mechanism is contractual, not discovered. Named limits: only the coarse ranking resolves — on the honest *t* one pair clears |*t*| = 2, the two 22-bp pairs sit at 1.9 and the four sub-10-bp gaps at |*t*| ≤ 0.6 against a 22.5 bp/yr floor; the early era is not significant alone; the fee sheet is an ASSUMPTION carrying hindsight and the headline pair is a selection off it; the cohort is five survivors.
- **Tradability — Investable.** Not because the tape pins the number down — the honest interval is 10 to 42 bp — but because the act is a **purchase decision with no forecast, no timing, no turnover and no capacity constraint** on the wrapper that wins, and because even at the bottom of that interval the cheap wrapper is not the worse buy — a dominance argument, not a forecast (GLDM turns $433m a day and repays a 30 bp execution penalty in 418 days), worth **2.6% of terminal wealth over ten years**. It is *not* a reason to sell an appreciated position (26–75 years to repay the tax), *not* a long/short (dead above ~18 bp of borrow), and *not* an argument for the fee-cheapest name at size (BAR: $21m/day, 4.8 days of ADV for a $100m ticket).